In [0]:
data=[(1,"Yash"),(2,"Ravi"),(3,"Ravipo"),(4,"Kup")]

schema="id int, name string"

df=spark.createDataFrame(data,schema)

df.display()




---

## Steps to Access Azure Data Lake from Databricks Using a Service Principal

### **Step 1: Create a Service Principal (Application)**

We need a Service Principal so **Azure Databricks** can access the storage account.

1. Go to **Azure Portal**
2. Navigate to **Microsoft Entra ID**
3. Click **App registrations**
4. Click **New registration**
5. Enter an application name and create it
6. After creation:

   * Copy **Application (Client) ID**
   * Copy **Directory (Tenant) ID**

**Create Client Secret**

1. Open the created application
2. Go to **Certificates & secrets**
3. Click **New client secret**
4. Give it a name and create
5. **Copy the secret value** (this will be used in Databricks)

---

### **Step 2: Grant Storage Account Access to the Service Principal**

Now give storage access to the Service Principal.

1. Go to your **Azure Storage Account**
2. Click **Access Control (IAM)**
3. Click **Add → Add role assignment**
4. Select **Storage Blob Data Contributor**
5. Click **Next**
6. Select **Service Principal**
7. Search and select your application
8. Click **Review + Assign**

---

### **Step 3: Configure Databricks to Access Data Lake Using Service Principal**

Now configure Databricks to use the Service Principal credentials.

1. Open **Databricks Workspace**
2. Create a notebook
3. Use the Azure-provided OAuth configuration code for ADLS access
4. Replace:

   * **Client ID**
   * **Tenant ID**
   * **Client Secret**
   * **Storage Account Name**
   * **Container Name**

Example (conceptual):

```python
spark.conf.set("fs.azure.account.auth.type.<storage>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.<storage>.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<storage>.dfs.core.windows.net", "<CLIENT_ID>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<storage>.dfs.core.windows.net", "<CLIENT_SECRET>")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<storage>.dfs.core.windows.net",
               "https://login.microsoftonline.com/<TENANT_ID>/oauth2/token")
```

---

### ✅ **Final Result**

* Service Principal is created
* Storage account access is granted
* Databricks is authenticated using Service Principal
* Databricks can now **read/write data** from Azure Data Lake

---




In [0]:
#service principle




In [0]:
spark.conf.set("fs.azure.account.auth.type.godzilladatalake.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.godzilladatalake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.godzilladatalake.dfs.core.windows.net", "ef80a2f2-5eb7-4a43-a2d9-b9570ade48d3")
spark.conf.set("fs.azure.account.oauth2.client.secret.godzilladatalake.dfs.core.windows.net", dbutils.secrets.get(scope="yashscope", key="godzillasecret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.godzilladatalake.dfs.core.windows.net", "https://login.microsoftonline.com/c2ed6e7b-02c5-45a8-97f9-55aec4f64929/oauth2/token")

### dbutils.fs()

In [0]:
# now databrciks has acces to the storage account
# You can check the files stored in storage account

print(dbutils.fs.ls("abfss://source@godzilladatalake.dfs.core.windows.net"))
print(dbutils.fs.ls("abfss://destination@godzilladatalake.dfs.core.windows.net"))# destination container does not have any file



### dbutils.widget
We need to make our databrick notebook parametize like func to take input from users

In [0]:
dbutils.widgets.text("p_name","Yash") # take input from user by default it is yash


---

## Hiding Secrets in Databricks Using Azure Key Vault

To avoid **hardcoding secrets** (client secret, passwords) in Databricks notebooks, we use **Azure Key Vault**. Databricks securely fetches secrets from Key Vault using a **Secret Scope**.

---

## **Step 1: Create Azure Key Vault**

1. Go to **Azure Portal**
2. Search for **Key Vault**
3. Click **Create**
4. Enter:

   * Key Vault name
   * Resource group
   * Region
5. In **Access configuration**, select:

   * **Vault access policy**
6. Click **Create**

---

### **Assign Permission to Create Secrets**

Before adding secrets, you must have permission.

1. Open the created **Key Vault**
2. Go to **Access control (IAM)**
3. Click **Add role assignment**
4. Select **Key Vault Administrator**
5. Assign it to **your email ID**
6. Click **Assign**

---

### **Create a Secret in Key Vault**

1. Go to **Objects → Secrets**
2. Click **Generate/Import**
3. Enter:

   * **Secret Name** (example: `databricks-client-secret`)
   * **Secret Value** (your client secret)
4. Click **Create**

---

## **Step 2: Create Secret Scope in Databricks (Link Key Vault)**

Now we link Databricks to the Key Vault.

### **Create Secret Scope**

1. Go to **Azure Portal**
2. Open **Azure Databricks**
3. Copy the **Databricks Workspace URL**
4. Open browser and paste:

   ```
   https://<databricks-workspace-url>/#secrets/createScope
   ```
5. Fill details:

   * **Scope Name** (example: `yashscope`)
   * **DNS Name** → Key Vault **Vault URI**
   * **Resource ID** → Key Vault **Resource ID**

📍 To get these:

* Go to **Key Vault → Properties**
* Copy **Vault URI**
* Copy **Resource ID**

6. Click **Create**

---

## **Step 3: Use Secrets Securely in Databricks Notebook**

### **Verify Secret Scope**

```python
dbutils.secrets.list(scope="yashscope")
```

### **Fetch Secret Value**

```python
dbutils.secrets.get(scope="yashscope", key="databricks-client-secret")
```

✅ The secret value will **not be printed** in plain text.

---

## **Final Result**

* Secrets are stored securely in **Azure Key Vault**
* Databricks is connected via **Secret Scope**
* No secrets are hardcoded in notebooks
* Better security and compliance

---



In [0]:
# https://adb-7405609975122794.14.azuredatabricks.net/?o=7405609975122794#secrets/createScope     -> use this url to create scope


### dbutils.secrets
> To keep secrets secure and hidden from admins and developers, we store them in **Azure Key Vault**. In **Databricks**, we create a **secret scope** linked to the Key Vault and access secrets using `dbutils.secrets.get()`. This avoids hard-coding secrets in notebooks and ensures secure access at runtime.


In [0]:
#To ckeck the secret in secrect scope
print(dbutils.secrets.list(scope="yashscope"))

# To get the value of secret
print(dbutils.secrets.get(scope="yashscope", key="godzillasecret"))



### Data reading from azure cloud storage

In [0]:
#Data Reading

#Always run the congiguration notebook first for storage account access

sales_df=spark.read.format("csv")\
    .option("header","true")\
    .option("inferSchema","true")\
    .load("abfss://source@godzilladatalake.dfs.core.windows.net/")




In [0]:
display(sales_df)

### Pyspark transformation

In [0]:
#Data Transformation

from pyspark.sql.functions import *
from pyspark.sql.types import *





In [0]:
#Upadte thee vaules in a col
sales_df.withColumn("Item_Type",split(col("Item_Type")," ")).display()

In [0]:
#Create new col

sales_df.withColumn("NewCol",lit("Superman")).display()

In [0]:
# Cahnge the col data type

sales_df.withColumn("Item_Visibility",col("Item_Visibility").cast("String")).display()

### Delta Lake

Delta Lake is used to **reduce compute and improve performance**.

When data is stored as **Parquet**, Spark must scan **metadata from every Parquet file** (for example, 1000 files), which increases compute and time.

When we convert Parquet to **Delta Lake**:

* Data is still stored as **Parquet files**
* Delta adds a **_delta_log (transaction log)**

The transaction log stores **metadata, schema, file list, and statistics**, so Spark reads metadata from the **log instead of every file**, saving compute and making queries faster.

---



In [0]:
#Write the data
sales_df.write.format("delta")\
  .mode("append")\
  .option("path","abfss://destination@godzilladatalake.dfs.core.windows.net/sales")\
  .save()

### Managed Delta Table

- Metadata such as table definitions and data type definitions are stored in the **Metastore**.
- The actual data is stored in the **default cloud storage** managed by Databricks.
- Databricks fully manages both the **metadata and the data**.
- When the table is dropped, **both metadata and data are deleted automatically**.


### External Delta Table

- Metadata such as table definitions and data type definitions are stored in the **Metastore**.
- The actual data is stored in an **external cloud storage location** (ADLS / S3 / GCS).
- Databricks manages **only the metadata**, not the data.
- When the table is dropped, **only metadata is deleted**, and the data remains intact.


In [0]:
%sql
-- Create database
create database  sales_db;


In [0]:
%sql

--- Create table

create table sales_db.salesTable(
  id int,
  name varchar(20),
  age int
)

using delta

In [0]:
%sql

SHOW TABLES IN sales_db;

In [0]:
%sql 
---Insert data into table

insert into sales_db.salesTable values(1, "Yash",30), (2, "Rami",45),(3,"Pol",43);

In [0]:
%sql
select * from sales_db.salesTable;

In [0]:
%sql
--- If we delete the table it will be deleted from the databricks data lake also

drop table sales_db.salesTable;
    
show tables in sales_db;



### External Table

**`NO_PARENT_EXTERNAL_LOCATION_FOR_PATH`** in **Azure Databricks Unity Catalog**, with **what + why** clearly explained.

---

## Problem (Why this error comes)

In **Unity Catalog**, Databricks **does NOT allow direct access** to cloud storage paths.

👉 Every external cloud path **must be registered** as:

* **Storage Credential** (who can access)
* **External Location** (which path can be accessed)

If a table path is not under a registered external location, Databricks throws:

```
NO_PARENT_EXTERNAL_LOCATION_FOR_PATH
```

---

## Key Concepts (Very Important)

### 1️⃣ Storage Credential

* Defines **authentication**
* Uses **Managed Identity / Service Principal**
* Answers: **WHO is allowed to access storage**

### 2️⃣ External Location

* Maps:

  * **Cloud storage path** (ADLS / S3 / GCS)
  * **Storage Credential**
* Answers: **WHICH path can be accessed**

> External tables can be created **ONLY inside an External Location**

---

## Step-by-Step Solution (Azure + Databricks)

---

## STEP 1: Create Access Connector for Azure Databricks

**Purpose:**
This creates a **Managed Identity** that Databricks will use to access ADLS.

**Steps:**

1. Go to **Azure Portal**
2. Search **Access Connector for Azure Databricks**
3. Click **Create**
4. Provide:

   * Subscription
   * Resource Group
   * Name (e.g. `adb-access-connector`)
   * Region
5. Click **Create**

✅ Output: A managed identity is created

---

## STEP 2: Grant Storage Permission to Access Connector

**Purpose:**
Allow Databricks to **read/write data** in ADLS Gen2.

**Steps:**

1. Go to **ADLS Gen2 Storage Account**
2. Open **IAM (Access Control)**
3. Click **Add Role Assignment**
4. Select role:

   ```
   Storage Blob Data Contributor
   ```
5. Click **Next**
6. Select **Managed Identity**
7. Choose:

   * Resource type: *Access Connector for Azure Databricks*
   * Select your connector
8. Click **Assign**

✅ Databricks can now access the storage

---

## STEP 3: Create Storage Credential (Databricks)

**Purpose:**
Tell Unity Catalog **which identity** to use.

**Steps (UI):**

1. Go to **Databricks Workspace**
2. Open **Catalog Explorer**
3. Go to **External Data → Storage Credentials**
4. Click **Create**
5. Provide:

   * Name (e.g. `adls_credential`)
   * Identity type: **Managed Identity**
   * Access Connector Resource ID
6. Click **Create**

✅ Storage Credential is ready

---

## STEP 4: Create External Location (Databricks)

**Purpose:**
Register your **ADLS path** in Unity Catalog.

**Steps:**

1. Go to **Catalog Explorer**
2. Open **External Data → External Locations**
3. Click **Create**
4. Fill:

   * Name: `adls_external_location`
   * URL:

     ```
     abfss://container@storageaccount.dfs.core.windows.net/path/
     ```
   * Storage Credential: `adls_credential`
5. Click **Create**

✅ Path is now authorized in Unity Catalog

---

## STEP 5: Create External Table (Now it will work)

Now your table path is **inside a registered external location**, so no error.

```sql
CREATE TABLE catalog.schema.external_table
USING DELTA
LOCATION 'abfss://container@storageaccount.dfs.core.windows.net/path/table';
```

✅ Error resolved 🎉

---

## Summary Flow (Easy to Remember)

```
Access Connector (Managed Identity)
        ↓
IAM Role (Storage Blob Data Contributor)
        ↓
Storage Credential (WHO)
        ↓
External Location (WHICH PATH)
        ↓
External Table
```

---




In [0]:
%sql
--- create external table

create table sales_db.salesextable4(
  id int,
  name varchar(20),
  age int
)
using delta 
location "abfss://destination@godzilladatalake.dfs.core.windows.net/externalTable4"

In [0]:
%sql
insert into sales_db.salesextable4 values(1, "Yash",30), (2,"Rami",45),(3,"Pol",43);
    
select * from sales_db.salesextable4;

### Insert


In [0]:
%sql
insert into sales_db.salesextable4 values(4,"Ravi",80),(5,"Uo",23),(6,"Olk",34);

### Delete

It does not delete anything from the current parquet file it just create new parquet file with the removed record.




In [0]:
%sql
delete from sales_db.salesextable4 where id=5;  
select * from sales_db.salesextable4;

### 📂 Files created in Azure (Delta table)

part-00000-02ab71cd-....parquet  → INSERT id = 1, 2, 3  
part-00000-3bc49f11-....parquet  → INSERT id = 4, 5, 6  
deletion_vector_*.bin            → Created during DELETE id = 5  
_delta_log/                      → Transaction metadata

---

### 🧠 IMPORTANT RULE
👉 Databricks **never reads Parquet files directly**  
👉 It always reads Delta tables using **_delta_log**

---

### 🔄 How Databricks reads data after DELETE

#### 1️⃣ Read _delta_log
Delta reads the latest transaction log and learns:
- Which Parquet files are active
- Which files have **deletion vectors**
- Which rows are logically deleted

Example from log:
- File A → active (id 1,2,3)
- File B → active + deletion vector (id 4,5,6)
- Deletion vector → row index of id = 5 marked as deleted

---

#### 2️⃣ Read Parquet + Apply Deletion Vector
- Spark reads data from Parquet files
- Loads `deletion_vector_*.bin`
- Drops **id = 5 in memory**

---

### ✅ Final output returned to user
```

1, 2, 3, 4, 6

```




In [0]:
df = spark.read.format("delta").load(
  "abfss://destination@godzilladatalake.dfs.core.windows.net/externalTable4"
)
df.show()


### History


In [0]:
%sql

describe history sales_db.salesextable4;

### Time Travel
Roll back to any version

In [0]:
%sql
restore table sales_db.salesextable4 to version as of 2;

select * from sales_db.salesextable4;

## 🧹 VACUUM (Delta Lake)

VACUUM is used to **physically delete old Parquet files** (obsolete data files) from a Delta table.

After DELETE or UPDATE:
- Parquet files are **not removed immediately**
- They remain for **time travel and recovery**

---

### 🔹 Default behavior
```sql
VACUUM exdb.sales_extable;




### **VACUUM (Delta Lake)**

VACUUM is used to **physically delete old Parquet files (partitions)** from a Delta table.

```sql--
VACUUM exdb.sales_extable;
```

* This does **not delete files immediately**
* Default retention period is **7 days**

To delete Parquet files **immediately**:

```sql--
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM exdb.sales_extable RETAIN 0 HOURS;
```

⚠️ This permanently deletes files and disables time travel.

---




### delta Table optimization
Optimize- It combines the small small partitions into large partitions

👉 It rewrites your Delta table’s data files into fewer, larger files (file compaction).

Here’s exactly what it does, in your case.

🔧 What OPTIMIZE does (internals) 1️⃣ Reads _delta_log

Delta finds:

All active Parquet files

Files with deletion vectors

Small / fragmented files

2️⃣ Rewrites data into new Parquet files

Reads valid rows only (deleted rows excluded)

Writes new, larger Parquet files

Example (your case):

Before OPTIMIZE

file_A → [1] file_B → [2,3] (3 deleted via DV) file_C → empty DV → hides 3

After OPTIMIZE

file_NEW → [1,2]

3️⃣ Updates _delta_log

Marks old files as removed

Marks new file as active

Old files still exist physically (for time travel).

4️⃣ Speeds up reads 🚀

Fewer files to scan

No deletion vectors needed

Less metadata overhead

In [0]:
%sql 
optimize sales_db.salesextable4;
    
select * from sales_db.salesextable4;

### Z-ORDER BY (Delta Lake)

Z-ORDER BY organizes data **within partitions** to improve query performance.

It clusters related values together, similar to **indexing in SQL**, so Spark can read **less data** during queries.

Used with:

OPTIMIZE table_name ZORDER BY (column_name);


In [0]:
%sql 
optimize sales_db.salesextable4 zorder by(id);
    
select * from sales_db.salesextable4


### Incremental Loading using Auto Loader (Streaming DataFrame)

Auto Loader is used for **incremental data loading** in Databricks.

It automatically loads all existing files from the **source** to the **destination**.  
When a new file is uploaded to the source, Auto Loader **detects and loads only the new file**, not the old ones.

This is done using a **streaming DataFrame**, making ingestion scalable and efficient.

### Checkpointing in Streaming Queries

A streaming query needs to **store the current state of the stream** to track which files have already been read.

Databricks uses **checkpointing** during the write operation:
- Saves progress and offsets
- Ensures each file is processed **only once**
- Helps in **fault tolerance and recovery**

Without a checkpoint, the stream may reprocess old files.


📥 Incremental Loading using Auto Loader (Easy Explanation)
🔹 What this code does (in simple words)

Reads Parquet files incrementally from a source folder

Automatically detects new files

Loads only new data into the destination

Uses checkpointing to remember what is already processed

Writes data continuously into a Delta table

### 1️⃣ Read stream using Auto Loader

What happens here:

cloudFiles → enables Auto Loader

Reads Parquet files

schemaLocation → stores inferred schema (no re-inference every time)

Loads all existing files first, then only new files

In [0]:
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option(
      "cloudFiles.schemaLocation",
      "abfss://aldestination@godzilladatalake.dfs.core.windows.net/schema"
  ) \
  .load("abfss://alsource@godzilladatalake.dfs.core.windows.net")


### 2️⃣ Write stream to Delta (with checkpoint)

What happens here:

Writes streaming data to Delta format

checkpointLocation → saves stream state (which files are already read)

Ensures exactly-once processing

trigger(processingTime="5 seconds") → checks for new files every 5 seconds

Writes data into destination path

🧠 Why checkpoint is important

Tracks already processed files

Prevents duplicate data

Allows recovery if the job fails or restarts

In [0]:
df.writeStream \
  .format("delta") \
  .option(
      "checkpointLocation",
      "abfss://aldestination@godzilladatalake.dfs.core.windows.net/checkpoint"
  ) \
  .trigger(processingTime="5 seconds") \
  .start(
      "abfss://aldestination@godzilladatalake.dfs.core.windows.net/data"
  )
